In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,confusion_matrix)

### Part A: Probability & Conceptual Foundation (Theory)

**1. What is Conditional Probability?**
Conditional probability is the likelihood of an event occurring given that another event has already happened[cite: 1]. In message classification, it represents the probability that a message belongs to a specific class (e.g., Spam) given the presence of certain observed features (e.g., specific keywords or URLs)[cite: 1]. 

**2. Explain Bayes' Theorem and its importance in classification problems.**
Bayes' Theorem is a mathematical formula used to calculate conditional probability by updating our prior beliefs based on new evidence[cite: 1]. It is crucial in classification because it allows models to compute the probability of a class label given the input features, forming the foundation for probabilistic models like Naive Bayes[cite: 1].

**3. What assumptions does the Naive Bayes Classifier make?**
The Naive Bayes Classifier operates on the "naive" assumption of conditional independence[cite: 2]. It assumes that all input features are independent of one another given the target class, meaning the presence or value of one feature does not affect the probability of any other feature[cite: 2].

**4. Explain the working principle of:**
*   **K-Nearest Neighbors (KNN):** KNN classifies a new data point by measuring its distance to all training samples and identifying the 'K' closest neighbors[cite: 2]. The point is then assigned to the class that is most common among those nearest neighbors (majority voting)[cite: 2].
*   **Support Vector Machine (SVM):** SVM works by finding the optimal hyperplane that separates data points of different classes in a high-dimensional space[cite: 2]. It aims to maximize the "margin," which is the distance between the hyperplane and the closest data points from each class (the support vectors)[cite: 2, 3].

**5. Compare distance-based, probabilistic, and margin-based classifiers.**
Distance-based classifiers (like KNN) rely on geometric proximity and similarity metrics to group data[cite: 2]. Probabilistic classifiers (like Naive Bayes) use statistical likelihoods and Bayes' theorem to predict the most probable class[cite: 1, 2]. Margin-based classifiers (like SVM) focus on constructing optimal decision boundaries that maximize the separation space between different classes[cite: 2].

# Part B: Dataset Understanding & Preparation 

In [2]:
base_dir = Path().cwd().parent

dataset_path = base_dir/"Data"/"Message_Intelligence_Dataset_5200.csv"
model_dir = base_dir/"Model"

In [3]:
DF= pd.read_csv(dataset_path)
DF

,message_id,message_text,message_length,word_count,num_urls,num_digits,num_special_chars,spam_keyword_score,legit_keyword_score,sender_activity_score,sender_account_age_days,messages_sent_last_24h,timestamp,hour_of_day,day_of_week,spam_label
0,900001,Please find the attached invoice for the updat...,99,11,1,4,0,0,1,56.6,500.0,6.0,2025-11-24 02:00:00,2,0,0
1,900002,Let's catch up tomorrow regarding the timeline...,73,12,0,0,0,0,0,16.6,207.0,0.0,2025-12-17 21:00:00,21,2,0
2,900003,Can you send the report by end of day? next Mo...,67,13,0,0,0,0,1,25.7,418.0,6.0,2025-11-15 13:00:00,13,5,0
3,900004,Can you send the report by end of day? 10:30 A...,64,13,0,4,0,0,1,48.8,276.0,5.0,2025-12-17 23:00:00,23,2,0
4,900005,Could you review the document and share feedba...,84,14,0,0,0,0,1,33.0,683.0,7.0,2025-11-29 11:00:00,11,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5195,905196,Thanks for your help on the project update. th...,73,12,0,0,0,0,1,23.1,262.0,5.0,2025-10-28 13:00:00,13,1,0
5196,905197,Let's catch up tomorrow regarding the timeline...,108,12,1,4,0,0,0,62.4,398.0,6.0,2025-11-25 14:00:00,14,1,0
5197,905198,Here are the notes from today's discussion. th...,80,12,0,0,0,0,1,96.6,457.0,5.0,2025-10-29 14:00:00,14,2,0
5198,905199,Act fast! Your account will be suspended unles...,94,14,1,4,2,1,0,70.1,9.0,11.0,2025-11-20 23:00:00,23,3,1


In [4]:
DF.head()

,message_id,message_text,message_length,word_count,num_urls,num_digits,num_special_chars,spam_keyword_score,legit_keyword_score,sender_activity_score,sender_account_age_days,messages_sent_last_24h,timestamp,hour_of_day,day_of_week,spam_label
0,900001,Please find the attached invoice for the updat...,99,11,1,4,0,0,1,56.6,500.0,6.0,2025-11-24 02:00:00,2,0,0
1,900002,Let's catch up tomorrow regarding the timeline...,73,12,0,0,0,0,0,16.6,207.0,0.0,2025-12-17 21:00:00,21,2,0
2,900003,Can you send the report by end of day? next Mo...,67,13,0,0,0,0,1,25.7,418.0,6.0,2025-11-15 13:00:00,13,5,0
3,900004,Can you send the report by end of day? 10:30 A...,64,13,0,4,0,0,1,48.8,276.0,5.0,2025-12-17 23:00:00,23,2,0
4,900005,Could you review the document and share feedba...,84,14,0,0,0,0,1,33.0,683.0,7.0,2025-11-29 11:00:00,11,5,0


In [5]:
DF.isnull().sum()

message_id                   0
message_text                 0
message_length               0
word_count                   0
num_urls                     0
num_digits                   0
num_special_chars            0
spam_keyword_score           0
legit_keyword_score          0
sender_activity_score      106
sender_account_age_days    113
messages_sent_last_24h     162
timestamp                    0
hour_of_day                  0
day_of_week                  0
spam_label                   0
dtype: int64

In [6]:
DF.duplicated().sum()

np.int64(0)

In [7]:
DF.shape

(5200, 16)

In [8]:
DF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5200 entries, 0 to 5199
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   message_id               5200 non-null   int64  
 1   message_text             5200 non-null   object 
 2   message_length           5200 non-null   int64  
 3   word_count               5200 non-null   int64  
 4   num_urls                 5200 non-null   int64  
 5   num_digits               5200 non-null   int64  
 6   num_special_chars        5200 non-null   int64  
 7   spam_keyword_score       5200 non-null   int64  
 8   legit_keyword_score      5200 non-null   int64  
 9   sender_activity_score    5094 non-null   float64
 10  sender_account_age_days  5087 non-null   float64
 11  messages_sent_last_24h   5038 non-null   float64
 12  timestamp                5200 non-null   object 
 13  hour_of_day              5200 non-null   int64  
 14  day_of_week             

In [9]:
Target_Counts = DF["spam_label"].value_counts().reindex([0,1],fill_value=0)
Target_PT = (DF["spam_label"].value_counts(normalize=True).reindex([0,1],fill_value=0)*100).round(2)

Target_Summary = pd.DataFrame({
    "Message_Type":["Legitimete","Spam"],
    "Count":Target_Counts.values,
    "Percentage":[f"{x}%" for x in Target_PT.values]
})

display(Target_Summary)

,Message_Type,Count,Percentage
0,Legitimete,4227,81.29%
1,Spam,973,18.71%


In [10]:
Features = ['message_length', 'word_count', 'num_urls', 'num_digits', 
            'num_special_chars', 'spam_keyword_score', 'legit_keyword_score', 
            'sender_activity_score', 'sender_account_age_days', 
            'messages_sent_last_24h', 'hour_of_day', 'day_of_week']

Target = 'spam_label'

In [11]:
X = DF[Features]
y = DF[Target]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
Scaled_preprocessor = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

per_KNN_SVM=ColumnTransformer([
    ("columns",Scaled_preprocessor,Features)
])

NB_preprocessor = Pipeline([
    ("imputer",SimpleImputer(strategy="median"))
])

per_NB=ColumnTransformer([
    ("columns",NB_preprocessor,Features)])

In [14]:
def evaluate_model(model,X_test,y_test,model_name,pos_label=1):
    y_pred = model.predict(X_test)

    tn,fp,fn,tp = confusion_matrix(
    y_test,y_pred,labels=[0,1]).ravel()

    results={
    "Model":model_name,
    "Accuracy":accuracy_score(y_test,y_pred),
    "Precision":precision_score(y_test,y_pred,pos_label=pos_label,zero_division=0),
    "recall":recall_score(y_test,y_pred,pos_label=pos_label,zero_division=0),
    "F1":f1_score(y_test,y_pred,pos_label=pos_label,zero_division=0),
    "TN":tn,
    "FP":fp,
    "FN":fn,
    "TP":tp
    }

    return results

# Part C: Baseline Model – K-Nearest Neighbors 

In [15]:
Base_KNN = Pipeline([
    ("preprocesser",per_KNN_SVM),
    ("model",KNeighborsClassifier())
])

In [16]:
param_grid_knn = {
    "model__n_neighbors":[3,5,7,9],
    "model__weights":["uniform","distance"],
    "model__metric":["euclidean","manhattan"]
}

In [17]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_KNN = GridSearchCV(
    estimator=Base_KNN,
    param_grid=param_grid_knn,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True
)

grid_KNN.fit(X_train,y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocesser',
                                        ColumnTransformer(transformers=[('columns',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['message_length',
                                                                          'word_count',
                                                                          'num_urls',
                                                                          'num_digits',
                                                                          'num_special_chars',
                                                                          'spam_keyword_score',
                                                                          'legit_keyword_score',
                                                                          'sender_activity_score',
                                                                          'sender_account_age_days',
                                                                          'messages_sent_last_24h',
                                                                          'hour_of_day',
                                                                          'day_of_week'])])),
                                       ('model', KNeighborsClassifier())]),
             n_jobs=-1,
             param_grid={'model__metric': ['euclidean', 'manhattan'],
                         'model__n_neighbors': [3, 5, 7, 9],
                         'model__weights': ['uniform', 'distance']},
             scoring='f1')

In [18]:
cv_results_KNN = pd.DataFrame(grid_KNN.cv_results_)

distance_analysis = (cv_results_KNN[["param_model__n_neighbors","param_model__weights","param_model__metric","mean_test_score"]].sort_values("mean_test_score", ascending=False))

display(distance_analysis)

,param_model__n_neighbors,param_model__weights,param_model__metric,mean_test_score
11,5,distance,manhattan,1.000000
10,5,uniform,manhattan,1.000000
9,3,distance,manhattan,1.000000
13,7,distance,manhattan,1.000000
8,3,uniform,manhattan,1.000000
12,7,uniform,manhattan,0.999327
15,9,distance,manhattan,0.999327
0,3,uniform,euclidean,0.999322
1,3,distance,euclidean,0.999322
6,9,uniform,euclidean,0.998649


In [23]:
y_pred_KNN = best_KNN.predict(X_test)

misclassified = X_test.copy()

misclassified["Actual"] = y_test
misclassified["Predicted"] = y_pred_KNN

misclassified = misclassified[misclassified["Actual"] != misclassified["Predicted"]]

display(misclassified.head(10))

,message_length,word_count,num_urls,num_digits,num_special_chars,spam_keyword_score,legit_keyword_score,sender_activity_score,sender_account_age_days,messages_sent_last_24h,hour_of_day,day_of_week,Actual,Predicted


In [20]:
result_KNN = pd.DataFrame({
        "Model":"KNN",
        "Best Parameters":grid_KNN.best_params_,
        "Best CV F1":grid_KNN.best_score_
})
display(result_KNN)
best_KNN = grid_KNN.best_estimator_

,Model,Best Parameters,Best CV F1
model__metric,KNN,manhattan,1.0
model__n_neighbors,KNN,3,1.0
model__weights,KNN,uniform,1.0


In [22]:
test_result = evaluate_model(best_KNN,X_test,y_test,"KNN")

test_df_KNN = pd.DataFrame(test_result,index=[0])

display(test_df_KNN)

,Model,Accuracy,Precision,recall,F1,TN,FP,FN,TP
0,KNN,1.0,1.0,1.0,1.0,809,0,0,231


# Part D: Support Vector Machine Classifier

In [24]:
Base_svm = Pipeline([
    ("preprocesser",per_KNN_SVM),
    ("model",SVC())
])

In [25]:
param_grid_SVC = {
    "model__C":[0.1,1,10,100],
    "model__kernel":["linear","rbf"],
    "model__gamma":["scale","auto"]
}

In [26]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_SVC = GridSearchCV(
    estimator=Base_svm,
    param_grid=param_grid_SVC,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True
)

grid_SVC.fit(X_train,y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocesser',
                                        ColumnTransformer(transformers=[('columns',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['message_length',
                                                                          'word_count',
                                                                          'num_urls',
                                                                          'num_digits',
                                                                          'num_special_chars',
                                                                          'spam_keyword_score',
                                                                          'legit_keyword_score',
                                                                          'sender_activity_score',
                                                                          'sender_account_age_days',
                                                                          'messages_sent_last_24h',
                                                                          'hour_of_day',
                                                                          'day_of_week'])])),
                                       ('model', SVC())]),
             n_jobs=-1,
             param_grid={'model__C': [0.1, 1, 10, 100],
                         'model__gamma': ['scale', 'auto'],
                         'model__kernel': ['linear', 'rbf']},
             scoring='f1')

In [27]:
result_SVC = pd.DataFrame({
        "Model":"SVC",
        "Best Parameters":grid_SVC.best_params_,
        "Best CV F1":grid_SVC.best_score_
})
display(result_SVC)
best_SVC = grid_SVC.best_estimator_

,Model,Best Parameters,Best CV F1
model__C,SVC,0.1,1.0
model__gamma,SVC,scale,1.0
model__kernel,SVC,linear,1.0


In [28]:
svc_model = best_SVC.named_steps["model"]

print("Best SVM Parameters:", grid_SVC.best_params_)
print("Number of Support Vectors:", svc_model.n_support_)
print("Total Support Vectors:", svc_model.n_support_.sum())
print("Number of Support Vectors per Class:", svc_model.n_support_.tolist())

Best SVM Parameters: {'model__C': 0.1, 'model__gamma': 'scale', 'model__kernel': 'linear'}
Number of Support Vectors: [8 9]
Total Support Vectors: 17
Number of Support Vectors per Class: [8, 9]


In [29]:
test_result = evaluate_model(best_SVC,X_test,y_test,"SVC")

test_df_SVC = pd.DataFrame(test_result,index=[0])

display(test_df_SVC)

,Model,Accuracy,Precision,recall,F1,TN,FP,FN,TP
0,SVC,1.0,1.0,1.0,1.0,809,0,0,231


In [30]:
comparison = pd.concat([test_df_KNN,test_df_SVC],ignore_index=True)
display(comparison.sort_values(by="F1",ascending=False))

,Model,Accuracy,Precision,recall,F1,TN,FP,FN,TP
0,KNN,1.0,1.0,1.0,1.0,809,0,0,231
1,SVC,1.0,1.0,1.0,1.0,809,0,0,231


# Part E: Naive Bayes Classifier & Probability 

In [31]:
Base_NB = Pipeline([
    ("preprocesser",per_NB),
    ("model",GaussianNB())
])

In [33]:
param_grid_NB = {"model__var_smoothing":[1e-11,1e-10,1e-9,1e-8,1e-7]}

In [34]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_NB = GridSearchCV(
    estimator=Base_NB,
    param_grid=param_grid_NB,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True
)

grid_NB.fit(X_train,y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocesser',
                                        ColumnTransformer(transformers=[('columns',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median'))]),
                                                                         ['message_length',
                                                                          'word_count',
                                                                          'num_urls',
                                                                          'num_digits',
                                                                          'num_special_chars',
                                                                          'spam_keyword_score',
                                                                          'legit_keyword_score',
                                                                          'sender_activity_score',
                                                                          'sender_account_age_days',
                                                                          'messages_sent_last_24h',
                                                                          'hour_of_day',
                                                                          'day_of_week'])])),
                                       ('model', GaussianNB())]),
             n_jobs=-1,
             param_grid={'model__var_smoothing': [1e-11, 1e-10, 1e-09, 1e-08,
                                                  1e-07]},
             scoring='f1')

In [35]:
result_NB = pd.DataFrame({
        "Model":"GaussianNB",
        "Best Parameters":grid_NB.best_params_,
        "Best CV F1":grid_NB.best_score_
})
display(result_NB)
best_NB = grid_NB.best_estimator_

,Model,Best Parameters,Best CV F1
model__var_smoothing,GaussianNB,1.000000e-11,1.0


In [36]:
test_result = evaluate_model(best_NB,X_test,y_test,"GaussianNB")

test_df_NB = pd.DataFrame(test_result,index=[0])

display(test_df_NB)

,Model,Accuracy,Precision,recall,F1,TN,FP,FN,TP
0,GaussianNB,1.0,1.0,1.0,1.0,809,0,0,231


# Part F: Model Comparison & Evaluation 

In [37]:
all_model_metric = pd.concat([test_df_KNN,test_df_SVC,test_df_NB],ignore_index=True)
display(all_model_metric)

,Model,Accuracy,Precision,recall,F1,TN,FP,FN,TP
0,KNN,1.0,1.0,1.0,1.0,809,0,0,231
1,SVC,1.0,1.0,1.0,1.0,809,0,0,231
2,GaussianNB,1.0,1.0,1.0,1.0,809,0,0,231


# Part G: Final Analysis & Reporting 

### Part G: Final Analysis & Reporting

**23. Final Report Summary**[cite: 3]

*   **Strengths and Weaknesses of Each Classifier**:
    *   **K-Nearest Neighbors (KNN)**: 
        *   *Strengths*: Intuitive, makes no underlying assumptions about data distribution, and adapts well to local data structures.
        *   *Weaknesses*: Computationally expensive at inference time (must calculate distance to all training samples), sensitive to irrelevant features, and requires feature scaling.
    *   **Support Vector Machine (SVM)**:
        *   *Strengths*: Highly effective in high-dimensional spaces and robust against overfitting when the margin is maximized. Linear SVM is highly interpretable.
        *   *Weaknesses*: RBF kernel can be memory-intensive and slower to train on very large datasets. It also operates as a "black box" compared to Naive Bayes.
    *   **Naive Bayes**:
        *   *Strengths*: Extremely fast to train and predict, highly scalable, and handles high-dimensional text data exceptionally well.
        *   *Weaknesses*: Assumes all features are conditionally independent (which is rarely true in real-world data).

*   **Impact of Probability Assumptions in Naive Bayes**:
    *   Naive Bayes assumes that the presence of one feature (e.g., a specific spam keyword) is independent of another feature (e.g., the number of URLs) given the class label. While this assumption is mathematically strict and often violated in human communication, the model still achieved perfect classification on this dataset. This indicates that the probability distributions of the individual features provide strong enough signals to override the naive independence assumption.

*   **Trade-offs Between Interpretability and Performance**:
    *   Typically, complex models (like SVM with RBF kernels) offer higher performance at the cost of interpretability, while simpler models (like Naive Bayes) are highly interpretable but may underperform on complex non-linear data. However, in this dataset, all models achieved 100% accuracy. Therefore, the trade-off shifts entirely toward prioritizing computational speed and interpretability over complex model architectures.

*   **Business Recommendation for Real-World Deployment**:
    *   For a real-time Message Intelligence System, inference speed and scalability are the highest priorities. Given that all models performed equally well, **Naive Bayes** is the primary recommendation. It requires minimal computational overhead, can process thousands of messages per second, and easily updates its probabilistic priors as new data arrives. **Linear SVM** serves as a strong secondary option if margin-based confidence scores are required by the security team.

---

In [38]:
import joblib

joblib.dump(grid_KNN.best_estimator_, model_dir/"best_knn.pkl")
joblib.dump(grid_SVC.best_estimator_, model_dir/"best_svm.pkl")
joblib.dump(grid_NB.best_estimator_, model_dir/"best_nb.pkl")

['C:\\Users\\mantr\\Downloads\\Spam Alert\\Model\\best_nb.pkl']